<div align='center'>
<h1 style='font-size:38px;'>Computing Spectral Energy Distributions</h1>
<h2 style='font-weight:400;'>Using the <code>SED</code> class for disk emission modeling</h2>
<p><em>From grain properties to multi-component SEDs</em></p>
</div>

---
**This tutorial covers:**
- Setting up the SED class with grain and star objects
- Defining disk geometry and size distribution parameters
- Computing thermal and scattered light SEDs
- Comparing model SEDs to observations
- Exploring how disk parameters affect the SED shape

In [ ]:
from pyGrater import Grain, Star, SED

from pyGrater.density import two_power_law
from pyGrater.size_distributions import power_law_distribution

import matplotlib.pyplot as plt
import numpy as np

## 1. Initialize grain and star objects
Load precomputed optical efficiencies and the stellar spectrum.

In [ ]:
grain = Grain(redo_Q=False, composition='astroSi')
star  = Star(star_name='HD113766')

print(f"Grain composition : {grain.grain_composition_name}")
print(f"Sublimation temp  : {grain.Tsub} K")
print(f"Star              : {star.star_name}")
print(f"  Teff            : {star.temp} K")
print(f"  Distance        : {star.distance} pc")
print(f"  Radius          : {star.radius} R_sun")

## 2. Define the wavelength grid
Choose wavelengths spanning from optical to sub-millimeter to capture all disk components.

In [ ]:
wavelengths = np.geomspace(0.5, 100.0, 200)   # µm

print(f"Wavelength range : {wavelengths.min():.2f} – {wavelengths.max():.1f} µm")
print(f"Number of points : {len(wavelengths)}")

## 3. Define disk and grain size parameters
Set up the physical model for the disk geometry and the dust size distribution.

In [ ]:
params = {
    # ── Disk geometry ──────────────────────────────────────────────────────
    'r0'       : 1.0,    # Reference radius [AU]
    'h0'       : 0.1,     # Scale height at r0 [AU]
    'alphain'  : 10.0,    # Inner power-law index (steep inner wall)
    'alphaout' : -4.0,    # Outer power-law index
    'gamma'    : 2.0,     # Vertical profile exponent
    'beta'     : 2.0,     # Scale height flaring exponent
    # ── Viewing geometry ───────────────────────────────────────────────────
    'itilt'    : 0.0,     # Inclination [deg]  (0 = face-on)
    'PA'       : 0.0,     # Position angle [deg]
    'omega'    : 0.0,     # Longitude of ascending node [deg]
    # ── Grain size distribution ────────────────────────────────────────────
    'a_min'             : 1e-7,    # Minimum grain size [m]
    'a_max'             : 1000e-6,  # Maximum grain size [m]
    'kappa'             : 3.5,      # Power-law index
    'N_sizes_integral'  : 200,      # Number of size bins for integration
    # ── Scattering ─────────────────────────────────────────────────────────
    'g'        : 0.5,     # Henyey-Greenstein asymmetry parameter
    # ── Disk mass ──────────────────────────────────────────────────────────
    'M_tot'    : 1e-3,    # Total dust mass [M_earth]
}

print("Disk parameters:")
for k, v in params.items():
    print(f"  {k:22s} = {v}")

## 4. Create the SED object and compute the SED
The `SED` class integrates the grain optical properties, stellar spectrum, and disk geometry.

In [ ]:
sed_obj = SED(grain, star, two_power_law, power_law_distribution, wavelengths, N_distances=400)

print("Computing SED …")
sed_therm, sed_sca = sed_obj.get_SED(keep_separate_fluxes=True, **params)
sed_total  = sed_therm + sed_sca

# Stellar photosphere interpolated onto SED grid
star_phot = np.interp(wavelengths, star.waves, star.flux, left=np.nan, right=np.nan)

print(f"Peak thermal flux    : {sed_therm.max():.3e} Jy  at {wavelengths[np.argmax(sed_therm)]:.1f} µm")
print(f"Peak scattered flux  : {sed_sca.max():.3e} Jy  at {wavelengths[np.argmax(sed_sca)]:.1f} µm")

## 5. Plot the full SED
Show the stellar photosphere, thermal emission, scattered light, and total disk flux.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

# ax.loglog(wavelengths, star_phot,  color='gold',    lw=2,   label='Stellar photosphere')
ax.loglog(wavelengths, sed_total,  color='black',   lw=2,   label='Disk total')
ax.loglog(wavelengths, sed_therm,  color='tomato',  lw=1.5, ls='--', label='Thermal emission')
ax.loglog(wavelengths, sed_sca,    color='steelblue', lw=1.5, ls=':', label='Scattered light')

ax.set_xlabel('Wavelength [µm]', fontsize=13)
ax.set_ylabel('Flux density [Jy]', fontsize=13)
ax.set_title(
    f'SED  –  {star.star_name}  |  r₀={params["r0"]} AU, '
    f'a_min={params["a_min"]*1e6:.0f} µm, κ={params["kappa"]:.1f}',
    fontsize=13,
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Effect of minimum grain size
Vary `a_min` to see how the smallest grains control the scattered-light and mid-IR emission.

In [ ]:
amin_values = [1e-6, 5e-6, 10e-6, 50e-6, 100e-6]   # metres

fig, ax = plt.subplots(figsize=(11, 6))

for amin in amin_values:
    p = dict(params, a_min=amin)
    th, sc = sed_obj.get_SED(keep_separate_fluxes=True, **p)
    total = th + sc
    ax.loglog(wavelengths, total, lw=2, label=f'a_min = {amin*1e6:.0f} µm')

ax.set_xlabel('Wavelength [µm]', fontsize=13)
ax.set_ylabel('Flux density [Jy]', fontsize=13)
ax.set_title('Effect of minimum grain size on the SED', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Effect of power-law index κ
Steeper size distributions (larger κ) favour small grains, boosting scattered light.

In [ ]:
kappa_values = [2.5, 3.0, 3.5, 4.0, 4.5]

fig, ax = plt.subplots(figsize=(11, 6))

for kappa in kappa_values:
    p = dict(params, kappa=kappa)
    th, sc = sed_obj.get_SED(keep_separate_fluxes=True, **p)
    total = th + sc
    ax.loglog(wavelengths, total, lw=2, label=f'κ = {kappa}')

ax.set_xlabel('Wavelength [µm]', fontsize=13)
ax.set_ylabel('Flux density [Jy]', fontsize=13)
ax.set_title('Effect of size-distribution index κ on the SED', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Effect of disk radius r₀
Moving the dust belt inward/outward shifts the thermal peak towards shorter/longer wavelengths.

In [ ]:
r0_values = [30, 50, 85, 130, 200]   # AU

fig, ax = plt.subplots(figsize=(11, 6))

for r0 in r0_values:
    p = dict(params, r0=r0)
    th, sc = sed_obj.get_SED(keep_separate_fluxes=True, **p)
    total = th + sc
    ax.loglog(wavelengths, total, lw=2, label=f'r₀ = {r0} AU')

ax.set_xlabel('Wavelength [µm]', fontsize=13)
ax.set_ylabel('Flux density [Jy]', fontsize=13)
ax.set_title('Effect of belt radius r₀ on the SED', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Thermal vs scattered light fraction
Understand which component dominates at each wavelength.

In [ ]:
frac_therm = sed_therm / np.where(sed_total > 0, sed_total, np.nan)
frac_sca   = sed_sca   / np.where(sed_total > 0, sed_total, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.loglog(wavelengths, sed_therm, color='tomato',    lw=2, label='Thermal')
ax.loglog(wavelengths, sed_sca,   color='steelblue', lw=2, label='Scattered')
ax.set_xlabel('Wavelength [µm]', fontsize=12)
ax.set_ylabel('Flux density [Jy]', fontsize=12)
ax.set_title('Thermal vs scattered flux', fontsize=12)
ax.legend(fontsize=10);  ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogx(wavelengths, frac_therm * 100, color='tomato',    lw=2, label='Thermal fraction')
ax.semilogx(wavelengths, frac_sca   * 100, color='steelblue', lw=2, label='Scattered fraction')
ax.axhline(50, color='k', ls='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Wavelength [µm]', fontsize=12)
ax.set_ylabel('Fraction of total disk flux [%]', fontsize=12)
ax.set_title('Thermal / scattered fraction', fontsize=12)
ax.set_ylim(0, 105)
ax.legend(fontsize=10);  ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Summary

- The `SED` class integrates grain optical properties, stellar spectrum, and disk geometry over grain sizes and disk distances.
- **Thermal emission** dominates at mid- to far-IR wavelengths and peaks at a wavelength set by the grain temperature (hence belt radius).
- **Scattered light** dominates at optical/near-IR wavelengths and depends strongly on the minimum grain size and the size-distribution slope κ.
- **`a_min`** controls the abundance of small, efficient scatterers — smaller `a_min` boosts near-IR scattered light.
- **κ** controls the relative weight of small vs large grains — steeper distributions (larger κ) favour small grains.
- **`r₀`** sets the characteristic grain temperature; a larger belt radius → cooler grains → longer-wavelength thermal peak.